# 第5章 ロボットの作業を計画させる (タスク分解・次の一手・手先の軌跡)

Cosmos 3 の Reasoner は、ロボットの「頭」として使うことを想定して学習されています (embodied reasoning)。
この章では、カメラ画像と作業の指示から、ロボットが何をすればよいかを答えさせます。

1. 作業を小さな手順に分けさせる (タスク分解)
2. 手順をプログラムで扱える JSON にする
3. 動画を見て「次の一手」を答えさせる
4. 指示の途中まで進んだ作業の続きを答えさせる
5. 手先 (グリッパー) が通る道筋を画像の座標で答えさせる (Action CoT)
6. 画像の座標を机の上の位置に直す (SO-101 などの実機につなぐ準備)
7. 自分のロボットのカメラ画像で試す

ここで出てくるのは「計画」までです。実機を動かすのは第11章 (方策モデル) と、各自のロボットの制御プログラムの役目です。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外。Ampere 以降の GPU と bf16 が前提です)。
Reasoner (文章で答える部分) だけを使うので、GPU メモリは 10GB 前後で足ります。

上から順に実行してください。「モデルを読み込む」セルは初回に 5GB ほどダウンロードします。

In [ ]:
# @title GPU を確かめる
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    print("GPU が見つかりません。Colab ならランタイムのタイプで L4 か A100 を選んでください。")

In [ ]:
# @title インストール (1〜2 分)
!pip install -q -U "transformers>=5.18" accelerate av huggingface_hub

In [ ]:
# @title サンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
# NVIDIA の公式クックブック (cosmos-framework) とモデルカードに置かれているお手本の素材
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title モデルを読み込む (初回は 5GB ほどダウンロード)
import json
import re

from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "nvidia/Cosmos3-Edge"
# Transformers は統合チェックポイントから Reasoner の部分だけを読み込む (生成用の部分は捨てる)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

In [ ]:
# @title 座標を画像に描く関数
from PIL import ImageFont

COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#bfef45"]


def to_px(x, y, img):
    """Cosmos の座標 (左上が原点、x・y それぞれ 0〜1000 に正規化) をピクセルに直す"""
    return x / 1000 * img.width, y / 1000 * img.height


def draw(img, items, path=False, width=3):
    """bbox_2d (枠) と point_2d (点) を描いた画像を返す。path=True なら点を順に線でつなぐ (軌跡)"""
    img = load_image(img).copy()
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, img.width // 45))
    items = items if isinstance(items, list) else [items]
    pts = []
    for i, it in enumerate(items):
        c = COLORS[i % len(COLORS)]
        label = str(it.get("label", i))
        if "bbox_2d" in it:
            x1, y1 = to_px(*it["bbox_2d"][:2], img)
            x2, y2 = to_px(*it["bbox_2d"][2:4], img)
            d.rectangle([x1, y1, x2, y2], outline=c, width=width)
            d.text((x1 + 4, y1 + 2), label, fill=c, font=font, stroke_width=2, stroke_fill="white")
        if "point_2d" in it:
            x, y = to_px(*it["point_2d"][:2], img)
            pts.append((x, y, label))
    if path and len(pts) > 1:
        d.line([(x, y) for x, y, _ in pts], fill="#ffd400", width=width + 1)
    r = max(5, img.width // 110)
    for j, (x, y, label) in enumerate(pts):
        c = "#ffd400" if path else COLORS[j % len(COLORS)]
        d.ellipse([x - r, y - r, x + r, y + r], fill=c, outline="black", width=2)
        text = str(j + 1) if path else label
        d.text((x + r + 3, y - r), text, fill=c, font=font, stroke_width=2, stroke_fill="black")
    return img

## 1. 作業を小さな手順に分けさせる

モデルカードのお手本です。考えるモード (`thinking=True`) で、考えた過程も見てみます。

In [ ]:
img = fetch(f"{HF}/example_reasoning_input.png")
display(IPImage(img, width=480))
plan = ask("The task is to put flower into the red bottle. Generate a plan consisting of subtasks for accomplish the task.",
           img, thinking=True, max_new_tokens=4096)
print("--- 考えた過程 (先頭) ---\n", ask.last_think[:1200])
print("\n--- 計画 ---\n", plan)

## 2. 手順を JSON にする

ロボットのプログラムに渡すなら、使える動作の種類を決めておき、その中から選ばせると扱いやすくなります。
ここでは SO-101 の制御でよく使う 5 種類 (`move_above` / `grasp` / `lift` / `place` / `release`) に限定します。

In [ ]:
SKILLS = ["move_above", "grasp", "lift", "place", "release"]
answer = ask(f"""The task is to put flower into the red bottle.
Plan the robot arm motions. Use only these skills: {SKILLS}.
Return a JSON list. Every item must have the keys "step", "skill" and "target", like this:
[{{"step": 1, "skill": "move_above", "target": "flower"}}, {{"step": 2, "skill": "grasp", "target": "flower"}}]""", img)
print(answer)
steps = parse_json(answer) or []
for s in steps:
    ok = "" if s.get("skill") in SKILLS else "  ← 決めた動作以外"
    print(f"{s.get('step')}. {s.get('skill')}({s.get('target')}){ok}")

## 3. 動画を見て「次の一手」を答えさせる

2 台のアームが箱詰めをしている動画です。ここまでの様子から、次に何をするかを答えさせます。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/robotics_next_action.mp4")
display(Video(video, embed=True, width=480))
answer = ask("What can be the next immediate action?", video, thinking=True)
print("--- 考えた過程 (先頭) ---\n", ask.last_think[:1200])
print("\n--- 次の一手 ---\n", answer)

## 4. 作業の続きを答えさせる

全体の作業と、今やっている手順を渡して、次の手順を答えさせます (プリンターのインク交換)。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/assisted_task_next_action.mp4")
display(Video(video, embed=True, width=480))
answer = ask('''This is the overall task that the agent is trying to complete: "The student exchanges the black ink cartridge of the printer."
In the video, the agent is trying to follow the instruction (a single step out of many to complete the overall task): "place old ink_cartridge."
What should be the next action of the agent?''', video, thinking=True, max_new_tokens=4096)
print("--- 考えた過程 (先頭) ---\n", ask.last_think[:800])
print("\n--- 次の手順 ---\n", answer)

## 5. 手先が通る道筋を座標で答えさせる (Action CoT)

作業を言葉で考えたうえで、グリッパーが通る点の並び (`point_2d` のリスト) を答えさせます。
座標は第3章と同じ、縦横それぞれ 0〜1000 です。黄色い線が答えの道筋、番号が通る順番です。

ここでは公式のプロンプトガイドと同じく、**考える書式 (`<think>` … `</think>`) の指示を質問文の中に書き**、`thinking=False` で聞きます。
`thinking=True` (チャットテンプレートの考えるモード) で聞くよりも、Edge ではこちらのほうが道筋が安定しました
(`thinking=True` だと、点が 1 か所に固まった答えがよく返ってきます)。

In [ ]:
TRAJ_PROMPT = ('You are given the task "{task}". Specify the 2D trajectory your end effector should follow in pixel space. '
               'Return the trajectory coordinates in JSON format like this: {{"point_2d": [x, y], "label": "gripper trajectory"}}.\n'
               'Answer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\n'
               'Write your final answer immediately after the </think> tag.')

for img, task in [
    (fetch(f"{FW}/reasoner/assets/action_cot_trajectory.png"), "Move the pink bowl to the right"),
    (fetch(f"{HF}/example_reasoning_input.png"), "Put flower into the red bottle"),
]:
    answer = ask(TRAJ_PROMPT.format(task=task), img)
    traj = [p for p in parse_json(answer) or [] if isinstance(p, dict) and "point_2d" in p]
    print(f"[{task}]", [p["point_2d"] for p in traj])
    display(draw(img, traj, path=True))

## 6. 画像の座標を机の上の位置に直す

カメラが固定されていて机が平らなら、画像の点と机の上の点は**射影変換 (ホモグラフィ)** で 1 対 1 に対応します。
机の上の 4 点について「画像のどこに映るか」と「机のどこか (cm)」を測っておけば、Cosmos が答えた点を机の座標に直せます。

下の 4 点は**説明用の仮の値**です。自分のロボットでは、机に印を 4 つ置いて実測した値に書き換えてください。
机の座標まで直せれば、あとは SO-101 の逆運動学 (または保存済みのポーズ) で関節角に変換できます。

In [ ]:
import numpy as np


def homography(src, dst):
    """4 組以上の対応点から、src → dst の射影変換行列 (3x3) を求める"""
    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    _, _, vt = np.linalg.svd(np.array(A, float))
    return vt[-1].reshape(3, 3) / vt[-1, -1]


def apply_h(H, x, y):
    u, v, w = H @ [x, y, 1.0]
    return u / w, v / w


# 仮の校正値: 画像の座標 (0〜1000) と、その点の机の上の位置 (ロボットの根元を原点に cm)
image_pts = [(250, 900), (750, 900), (680, 450), (320, 450)]
table_cm = [(-15, 10), (15, 10), (15, 40), (-15, 40)]
H = homography(image_pts, table_cm)

for p in traj:
    x, y = p["point_2d"][:2]
    u, v = apply_h(H, x, y)
    print(f"画像 ({x:4}, {y:4}) → 机の上 ({u:6.1f} cm, {v:6.1f} cm)")

## 7. 自分のロボットのカメラ画像で試す

SO-101 の俯瞰カメラなどで撮った画像と、作業の指示 (英語) を入れてください。手順の計画と手先の道筋を返します。

In [ ]:
# @title 自分の画像で試す (colab-only)
from google.colab import files

task = "Pick up the red block and put it in the box"  # @param {type:"string"}

up = files.upload()
path = next(iter(up))
plan = ask(f"The task is to {task[0].lower() + task[1:]}. Generate a plan consisting of subtasks for accomplish the task.", path)
print("--- 計画 ---\n", plan)
answer = ask(TRAJ_PROMPT.format(task=task), path)
display(draw(path, parse_json(answer) or [], path=True))

## うまくいかないとき

- **道筋の点がずれる**: Edge (4B) は大きいモデル (Nano / Super) より座標の精度が落ちます。点は「だいたいの経由地」と考え、最後の位置合わせは第3章の枠や点 (つかむ物そのものを指させる) で行うと安定します
- **計画に関係ない手順が混ざる**: 使える動作の種類を決めて選ばせる (2 の書き方) と減ります
- **考えるモードが長すぎる**: 計画だけなら `thinking=False` でも十分なことが多いです